# 00 · Preparar o ambiente

Este notebook deixa o ambiente do projeto pronto. Ele pode ser executado quantas vezes quiser sem estragar nada (é **idempotente**):

1. Cria o catálogo `growdev` e os schemas `bronze`, `silver` e `gold`.
2. Cria o volume `growdev.bronze.landing`, a "caixa de entrada" dos arquivos brutos.
3. Copia os arquivos Parquet da pasta `dados/` do repositório para o volume. Um arquivo que já está lá não é copiado de novo.

> Execute este notebook uma vez antes das aulas. Ele também vira a primeira tarefa do job do projeto.

In [ ]:
dbutils.widgets.text("catalogo", "growdev", "Catálogo")
CATALOGO = dbutils.widgets.get("catalogo")
print(f"Catálogo do projeto: {CATALOGO}")

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronze.landing")
print("Catálogo, schemas e volume prontos")

In [ ]:
import os
import shutil

# O notebook roda dentro da pasta Projeto/; os dados ficam em ../dados
PASTA_DADOS = os.path.abspath("../dados")
VOLUME_LANDING = f"/Volumes/{CATALOGO}/bronze/landing"

tabelas = ["customers", "accounts", "account_balances", "loans"]

for tabela in tabelas:
    origem = f"{PASTA_DADOS}/{tabela}/{tabela}.parquet"
    destino_pasta = f"{VOLUME_LANDING}/{tabela}"
    destino = f"{destino_pasta}/{tabela}.parquet"

    os.makedirs(destino_pasta, exist_ok=True)
    if os.path.exists(destino):
        print(f"Já existe, não copiei: {destino}")
    else:
        shutil.copy(origem, destino)
        print(f"Copiado: {destino}")

In [ ]:
for tabela in tabelas:
    arquivos = dbutils.fs.ls(f"{VOLUME_LANDING}/{tabela}")
    print(f"{tabela}: {len(arquivos)} arquivo(s)")